# Chapter 7: Building Chat Applications
## OpenAI API Quickstart

This notebook is adapted from the [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) that includes notebooks that access [Azure OpenAI](aoai-assignment.ipynb) services.

The Python OpenAI API works with Azure OpenAI Models as well, with a few modifications. Learn more about the differences here: [How to switch between OpenAI and Azure OpenAI endpoints with Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# ภาพรวม  
"โมเดลภาษาขนาดใหญ่คือฟังก์ชันที่ทำการแมปข้อความเป็นข้อความ เมื่อได้รับข้อความนำเข้า โมเดลภาษาขนาดใหญ่จะพยายามทำนายข้อความที่จะตามมา"(1) โน้ตบุ๊ก 'เริ่มต้นอย่างรวดเร็ว' นี้จะแนะนำผู้ใช้ให้รู้จักกับแนวคิด LLM ในระดับสูง ความต้องการแพ็กเกจหลักสำหรับการเริ่มต้นใช้งาน AML การแนะนำอย่างนุ่มนวลเกี่ยวกับการออกแบบคำสั่งกระตุ้น และตัวอย่างสั้นๆ หลายตัวอย่างของการใช้งานที่แตกต่างกัน  


## สารบัญ  

[ภาพรวม](#overview)  
[วิธีการใช้บริการ OpenAI](#how-to-use-openai-service)  
[1. การสร้างบริการ OpenAI ของคุณ](#1.-creating-your-openai-service)  
[2. การติดตั้ง](#2.-installation)    
[3. ข้อมูลรับรอง](#3.-credentials)  

[กรณีการใช้งาน](#use-cases)    
[1. สรุปข้อความ](#1.-summarize-text)  
[2. การจัดประเภทข้อความ](#2.-classify-text)  
[3. สร้างชื่อผลิตภัณฑ์ใหม่](#3.-generate-new-product-names)  
[4. ปรับแต่งตัวจัดประเภท](#4.fine-tune-a-classifier)  

[ข้อมูลอ้างอิง](#references)


### สร้างพรอมต์แรกของคุณ  
แบบฝึกหัดสั้นๆ นี้จะให้การแนะนำพื้นฐานสำหรับการส่งพรอมต์ไปยังโมเดล OpenAI สำหรับงานง่ายๆ คือ "การสรุป"  


**ขั้นตอน**:  
1. ติดตั้งไลบรารี OpenAI ในสภาพแวดล้อม Python ของคุณ  
2. โหลดไลบรารีช่วยเหลือมาตรฐานและตั้งค่าข้อมูลรับรองความปลอดภัยของ OpenAI ตามปกติสำหรับบริการ OpenAI ที่คุณสร้างขึ้น  
3. เลือกโมเดลสำหรับงานของคุณ  
4. สร้างพรอมต์ง่ายๆ สำหรับโมเดล  
5. ส่งคำขอของคุณไปยัง API ของโมเดล!


### 1. ติดตั้ง OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. นำเข้าไลบรารีช่วยเหลือและสร้างข้อมูลประจำตัว (credentials)


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. การค้นหาโมเดลที่เหมาะสม  
โมเดลอย่าง GPT-4o และ GPT-4o mini สามารถเข้าใจและสร้างภาษาธรรมชาติได้ และมีให้ใช้งานบนแพลตฟอร์ม OpenAI โดยมีระดับพลังและความเร็วที่แตกต่างกัน เหมาะสมกับงานที่หลากหลาย  


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. การออกแบบพรอมต์  

"เวทมนตร์ของโมเดลภาษาขนาดใหญ่คือด้วยการฝึกฝนให้ลดข้อผิดพลาดในการทำนายเหล่านี้ผ่านข้อความจำนวนมหาศาล โมเดลจะได้เรียนรู้แนวคิดที่เป็นประโยชน์สำหรับการทำนายเหล่านี้ ตัวอย่างเช่น โมเดลเรียนรู้แนวคิดเช่น"(1):

* วิธีการสะกดคำ
* วิธีการทำงานของไวยากรณ์
* วิธีการเขียนคำพูดซ้ำโดยใช้สำนวนอื่น
* วิธีตอบคำถาม
* วิธีการสนทนา
* วิธีเขียนในหลายภาษา
* วิธีเขียนโค้ด
* เป็นต้น

#### วิธีควบคุมโมเดลภาษาขนาดใหญ่  
"จากอินพุตทั้งหมดสำหรับโมเดลภาษาขนาดใหญ่ ข้อความพรอมต์เป็นอินพุตที่มีอิทธิพลมากที่สุด(1)

โมเดลภาษาขนาดใหญ่สามารถถูกกระตุ้นให้สร้างผลลัพธ์ได้หลายวิธี:

คำสั่ง: บอกโมเดลว่าคุณต้องการอะไร
การเติมข้อความ: กระตุ้นให้โมเดลเติมสิ่งที่คุณต้องการเริ่มต้นไว้
การสาธิต: แสดงให้โมเดลเห็นสิ่งที่คุณต้องการ โดยใช้:
ตัวอย่างไม่กี่ตัวในพรอมต์
หลายร้อยหรือหลายพันตัวอย่างในชุดข้อมูลฝึกปรับแต่งเฉพาะ"



#### มีหลักการพื้นฐานสามข้อในการสร้างพรอมต์:

**แสดงและบอก**. ทำให้ชัดเจนว่าคุณต้องการอะไร ไม่ว่าจะผ่านคำสั่ง ตัวอย่าง หรือการผสมผสานของทั้งสองอย่าง หากคุณต้องการให้โมเดลจัดอันดับรายการตามลำดับตัวอักษรหรือจัดประเภทย่อหน้าตามอารมณ์ความรู้สึก ให้แสดงให้เห็นว่านั่นคือสิ่งที่คุณต้องการ

**จัดเตรียมข้อมูลคุณภาพ**. หากคุณพยายามสร้างตัวจัดประเภทหรือให้โมเดลทำตามรูปแบบ ให้แน่ใจว่ามีตัวอย่างเพียงพอ ตรวจสอบตัวอย่างของคุณให้ดี — โมเดลมักฉลาดพอที่จะมองข้ามข้อผิดพลาดสะกดพื้นฐานและให้คำตอบแก่คุณ แต่บางครั้งโมเดลอาจถือว่านั่นเป็นเจตนา ซึ่งอาจส่งผลต่อคำตอบ

**ตรวจสอบการตั้งค่าของคุณ**. การตั้งค่า temperature และ top_p ควบคุมว่าการตอบสนองของโมเดลจะมีความแน่นอนแค่ไหน หากคุณต้องการคำตอบที่ถูกต้องเพียงคำตอบเดียว คุณควรตั้งค่านี้ให้น้อยลง หากคุณต้องการคำตอบที่หลากหลายขึ้น คุณอาจตั้งค่าสูงขึ้น ความผิดพลาดอันดับหนึ่งที่คนมักทำกับการตั้งค่าเหล่านี้คือคิดว่ามันเป็นตัวควบคุม "ความเฉลียวฉลาด" หรือ "ความคิดสร้างสรรค์"


แหล่งที่มา: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. ส่ง! 


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### เรียกใช้งานเดิมซ้ำ ผลลัพธ์เปรียบเทียบกันอย่างไร?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## สรุปข้อความ  
#### ความท้าทาย  
สรุปข้อความโดยการเพิ่ม 'tl;dr:' ที่ส่วนท้ายของข้อความ สังเกตว่ารูปแบบโมเดลเข้าใจวิธีการทำงานหลาย ๆ อย่างโดยไม่ต้องมีคำแนะนำเพิ่มเติม คุณสามารถทดลองใช้ prompt ที่อธิบายได้มากกว่าคำว่า tl;dr เพื่อปรับเปลี่ยนพฤติกรรมของโมเดลและปรับแต่งการสรุปที่คุณได้รับ(3)  

งานวิจัยล่าสุดได้แสดงให้เห็นถึงความก้าวหน้าอย่างมีนัยสำคัญในงานและเกณฑ์มาตรวัดหลายประเภทของ NLP โดยการฝึกก่อนบนชุดข้อความขนาดใหญ่ก่อนจะปรับแต่งเฉพาะเรื่องในงานหนึ่ง ๆ แม้ว่าสถาปัตยกรรมโดยทั่วไปจะไม่ขึ้นกับงานเฉพาะ แต่วิธีนี้ยังคงต้องการชุดข้อมูลการปรับแต่งที่มีตัวอย่างเป็นพันหรือล้านตัวอย่าง ในทางตรงกันข้าม มนุษย์โดยทั่วไปสามารถทำงานภาษาใหม่ ๆ ได้จากตัวอย่างเพียงไม่กี่ตัวอย่างหรือคำแนะนำง่าย ๆ ซึ่งเป็นสิ่งที่ระบบ NLP ปัจจุบันยังมีปัญหาอยู่มาก ที่นี่เราแสดงให้เห็นว่าการเพิ่มขนาดโมเดลภาษาอย่างมากช่วยปรับปรุงประสิทธิภาพของงานที่ไม่ขึ้นกับงานและตัวอย่างน้อยได้อย่างยิ่ง บางครั้งถึงขั้นแข่งขันได้กับวิธีการปรับแต่งแบบสภาพศิลปะก่อนหน้านี้  



สรุป  


# แบบฝึกหัดสำหรับกรณีการใช้งานหลายอย่าง  
1. สรุปข้อความ  
2. จำแนกประเภทข้อความ  
3. สร้างชื่อผลิตภัณฑ์ใหม่


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## การจำแนกข้อความ  
#### ความท้าทาย  
จำแนกสิ่งของเป็นหมวดหมู่ที่กำหนดในเวลาอนุมาน ในตัวอย่างต่อไปนี้ เราให้ทั้งหมวดหมู่และข้อความที่จะจำแนกในพรอมต์(*playground_reference)  

คำถามจากลูกค้า: สวัสดี คีย์หนึ่งบนแป้นพิมพ์แล็ปท็อปของฉันเพิ่งเสียและฉันต้องการอะไหล่ทดแทน  

หมวดหมู่ที่จำแนก:  


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## สร้างชื่อผลิตภัณฑ์ใหม่
#### ความท้าทาย
สร้างชื่อผลิตภัณฑ์จากคำตัวอย่าง ที่นี่เรารวมข้อมูลเกี่ยวกับผลิตภัณฑ์ที่เราจะสร้างชื่อไว้ในคำสั่ง นอกจากนี้เรายังให้ตัวอย่างที่คล้ายกันเพื่อแสดงรูปแบบที่เราต้องการรับ เรายังตั้งค่าความร้อนสูงเพื่อเพิ่มความสุ่มและการตอบสนองที่สร้างสรรค์มากขึ้น

คำอธิบายผลิตภัณฑ์: เครื่องปั่นมิลค์เชคสำหรับใช้ที่บ้าน
คำหลัก: เร็ว, สุขภาพดี, ขนาดกะทัดรัด
ชื่อผลิตภัณฑ์: HomeShaker, Fit Shaker, QuickShake, Shake Maker

คำอธิบายผลิตภัณฑ์: รองเท้าคู่หนึ่งที่สามารถพอดีกับขนาดเท้าใดก็ได้
คำหลัก: ปรับได้, พอดี, omni-fit


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# เอกสารอ้างอิง  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [แนวทางปฏิบัติที่ดีที่สุดสำหรับการปรับแต่งโมเดล GPT เพื่อจำแนกข้อความ](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# สำหรับความช่วยเหลือเพิ่มเติม  
[ทีมพาณิชย์ OpenAI](AzureOpenAITeam@microsoft.com) 


# ผู้ร่วมเขียน  
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
